In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

**1. Quantization**

In [ ]:
# Dynamic quantization (easiest, good for RNNs/LSTMs)
quantized_model = torch.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)

In [ ]:
# Static quantization (better performance, for CNNs)
model.qconfig = torch.quantization.get_default_qconfig('fbgemm')
torch.quantization.prepare(model, inplace=True)

In [ ]:
# Calibrate with sample data
for data in calibration_dataset:
    model(data)

torch.quantization.convert(model, inplace=True)

In [ ]:
# Quantization-Aware Training (QAT) - best accuracy
model.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')
torch.quantization.prepare_qat(model, inplace=True)

In [ ]:
# Train normally, quantization is simulated
train_model(model)
torch.quantization.convert(model, inplace=True)

**2. Model Pruning**

In [2]:
import torch.nn.utils.prune as prune

In [ ]:
for module in model.modules():
    if isinstance(module, nn.Linear):
        prune.l1_unstructured(module, name='weight', amount=0.2)
        prune.remove(module, 'weight')

**3. ONNX Export and Optimization**

In [ ]:
import torch.oonx
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

In [ ]:
# Export to ONNX
dummy_input = torch.randint(0, vocab_size, (1, 128))
torch.oonx.export(
    model,
    dummy_input,
    'model.onnx',
    input_names=['input_ids'],
    output_names=['logits'],
    dynamic_axes={'input_ids': {0: 'batch_size'}},
    opset_version=11
)

In [ ]:
# Optimize ONNX model
onnx_model = onnx.load('model.onnx')
onnx.checker.check_model(onnx_model)

quantize_dynamic(
    'model.onnx',
    'model_quantized.onnx',
    weight_type=QuantType.QUInt8
)

ONNX Runtime (Cross-Platform Inference):

In [ ]:
import onnxruntime as ort

In [ ]:
# Load optimized ONNX model
session = ort.InferenceSession('model_quantized.onnx')

# Faster inference than PyTorch
inputs = {'input_ids' : input_data.numpy()}
outputs = session.run(None, inputs)